In [1]:
import pandas as pd

In [3]:
df_jobkorea = pd.read_json('./data/extracted_jobs_result_jobkorea.json')
df_saramin = pd.read_json('./data/extracted_jobs_result_saramin.json')
df_wanted = pd.read_json('./data/extracted_jobs_result_wanted.json')
len(df_jobkorea), len(df_saramin), len(df_wanted)

(10107, 25950, 4404)

In [9]:
df_jobkorea['연차/경력'].unique(), df_wanted['year_filter'].unique()

(array(['신입·경력', '경력10년↑', '경력3년↑', '경력7년↑', '신입·경력1년↑', '경력5년↑', '경력4년↑',
        '경력12년↑', '경력6년↑', '경력2년↑', '신입·경력3년↑', '경력8년↑', '경력무관', '경력1년↑',
        '경력', '경력9년↑', '경력15년↑', '신입', '신입·경력7년↑', '신입·경력2년↑', '신입·경력5년↑',
        '신입·경력4년↑', '경력11년↑', '신입·경력6년↑', '신입·경력9년↑', '신입·경력10년↑',
        '경력20년↑', '경력13년↑', None, '신입·경력8년↑', '', '신입·경력15년↑', '경력14년↑',
        '경력16년↑'], dtype=object),
 array([0.0, 1.0, 3.0, 5.0, ''], dtype=object))

In [ ]:
import pandas as pd
import re

def fill_and_format_year(row):
    val = row['year_filter']
    
    # 1. 이미 값이 채워져 있는 경우 (예: 0.0, 1.0, 3.0, 5.0)
    # 빈 문자열이 아니고, 결측치(NaN)가 아니라면 리스트로 묶어서 반환
    if pd.notna(val) and str(val).strip() != '':
        # (혹시 이미 리스트로 되어있다면 그대로 반환, 아니면 float형으로 변환 후 리스트 씌움)
        if isinstance(val, list):
            return val
        return [float(val)]
        
    # 2. 값이 비어있는 경우 ('') -> requirements 텍스트 분석 시작
    req_text = str(row['requirements']) 
    years = re.findall(r'(\d+)\s*년', req_text)
    
    if years:
        max_year = max([int(y) for y in years])
        
        if max_year >= 5:
            return [5.0]
        elif max_year >= 3:
            return [3.0]
        elif max_year >= 1:
            return [1.0]
            
    # 3. 숫자는 없지만 '신입'이라는 단어가 포함되어 있다면 [0.0] 처리
    if '신입' in req_text:
        return [0.0]
    
    # 4. 비어있는데 텍스트에서도 아무 힌트를 못 찾았다면?
    # 잡코리아에서 하셨던 것처럼 기본값으로 신입 [0.0] 취급
    return [0.0]

# 원본 보존을 위해 copy 후 적용
df_wanted_year_na = df_wanted.copy()
# DataFrame에 apply 함수를 사용해 행(row) 단위로 적
df_wanted_year_na['year_filter'] = df_wanted_year_na.apply(fill_and_format_year, axis=1)

# 결과 확인 (문자열로 변환하여 고유값 확인)
print("=== 변환 완료된 고유값 확인 ===")
print(df_wanted_year_na['year_filter'].astype(str).unique())

# (추천) 리스트 내부의 개별 고유값을 보고 싶다면 아래 방식 사용!
# print(df_wanted_year_na['year_filter'].explode().unique())

=== 변환 완료된 고유값 확인 ===
['[0.0]' '[1.0]' '[3.0]' '[5.0]']


In [19]:
df_1 = df_wanted_year_na[df_wanted_year_na['year_filter'] == '']
len(df_1)

0

In [ ]:
import pandas as pd
import re

def parse_jobkorea_multi_mapped(exp_str):
    # 1. 결측치 처리
    if pd.isna(exp_str) or str(exp_str).strip() == '':
        return [0.0]

    exp_str = str(exp_str)
    result = set()

    # 2. '신입', '무관' -> 0.0 추가
    if '신입' in exp_str or '무관' in exp_str:
        result.add(0.0)

    # 3. 정규식으로 숫자 추출 후 기준점에 맞춰 변환
    years = re.findall(r'(\d+)년', exp_str)
    
    if years:
        for y in years:
            y_int = int(y)
            # 7년이든 10년이든 5년 이상이면 모두 5.0으로 묶임
            if y_int >= 5:
                result.add(5.0)
            elif y_int >= 3:
                result.add(3.0)
            elif y_int >= 1:
                result.add(1.0)
    else:
        # 4. 숫자 없이 '경력'만 있는 경우 -> 1.0 추가
        if '경력' in exp_str:
            result.add(1.0) 

    # 최종 리스트 반환 (예: [0.0, 5.0])
    if result:
        return sorted(list(result))
    else:
        return [0.0]
df_jobkorea_year_na = df_jobkorea.copy()
# DataFrame에 적용
df_jobkorea_year_na['year_filter'] = df_jobkorea_year_na['연차/경력'].apply(parse_jobkorea_multi_mapped)

# ----------------- [테스트 결과 확인] -----------------
test_samples = [
    '신입·경력7년↑',   # 신입(0.0) + 7년(5.0)
    '신입·경력3년',    # 신입(0.0) + 3년(3.0)
    '경력12년↑',       # 12년(5.0)
    '경력무관',        # 무관(0.0)
    '신입·경력'        # 신입(0.0) + 단순경력(1.0)
]

print("=== 변환 결과 테스트 ===")
for sample in test_samples:
    print(f"{sample:10} ➡️ {parse_jobkorea_multi_mapped(sample)}")

=== 변환 결과 테스트 ===
신입·경력7년↑   ➡️ [0.0, 5.0]
신입·경력3년    ➡️ [0.0, 3.0]
경력12년↑     ➡️ [5.0]
경력무관       ➡️ [0.0, 1.0]
신입·경력      ➡️ [0.0, 1.0]


In [ ]:
print(df_jobkorea_year_na['year_filter'].astype(str).unique())

['[0.0, 1.0]' '[5.0]' '[3.0]' '[1.0]' '[0.0, 3.0]' '[0.0]' '[0.0, 5.0]']
